In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [28]:
# Extracting data from what the R script produced
cps_df = pd.read_csv("/Users/chrisgee/UCSD2526/replication_project/data/ipums_extracted_v3.csv")
cps_df = cps_df.set_index("STATEFIP")
cps_df

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,OCC50LY,INDLY,OCC90LY,IND90LY,FULLPART,FIRMSIZE,INCTOT,INCWAGE,EDATT,EDATTLY
STATEFIP,,,,,,,,,,,,,,,,,,,,,
6,1988,1,1,NaN,1.987100e+13,NaN,NaN,1,2901.0600,19871000000101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1988,1,1,NaN,1.987100e+13,NaN,NaN,2,2877.9600,19871000000102,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1988,1,1,NaN,1.987100e+13,NaN,NaN,3,3152.0400,19871000000103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,1988,2,1,NaN,1.987100e+13,NaN,NaN,1,1241.0400,19871000000301,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,1988,3,1,NaN,1.986100e+13,NaN,NaN,1,1922.3700,19861000000501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,1997,60517,12,299.0424,1.997091e+13,NaN,NaN,2,299.0424,19970906014702,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,1997,60517,12,299.0424,1.997091e+13,NaN,NaN,3,231.2252,19970906014703,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,1997,60517,12,299.0424,1.997091e+13,NaN,NaN,4,210.2731,19970906014704,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# Aggregating to find data points relevant to New Jersey and Pennsylvania
cps_df = cps_df.loc[[34, 42]]

# Creating identifier for states: 1 for NJ, 0 for PA
cps_df = cps_df.assign(NJ_PA = lambda x: x.index == 34).reset_index()
cps_df['NJ_PA'] = cps_df['NJ_PA'].astype(int)

# Creating a date time variable to easily work between dates and months
cps_df['DAY'] = 1
cps_df['PERIOD'] = pd.to_datetime(cps_df[['MONTH', 'YEAR', 'DAY']]).dt.to_period('M')
cps_df = cps_df.drop(columns=['DAY', 'STATEFIP'])

In [30]:
# Creating dummy variable for post policy period: 1 for after 04 1992, 0 for before 04 1992 (inclusive)
cps_df['POLICY_PERIOD'] = cps_df['PERIOD'].between('1992-05', '1997-12').astype(int)

# Creating a dummy variable for expasionary period: 1 for after  03 1991, 0 for before 03 1992 (inclusive), the recessionary period 
cps_df['RECESSION_PERIOD'] = cps_df['PERIOD'].between('1988-01', '1991-02').astype(int)

# To simplify the nuiances of employment status I converted the 'EMPSTAT' column into a binary value in accordance with the CPS variable code book
cps_df = cps_df[cps_df['EMPSTAT'].isin([10, 12, 20, 21, 22])]
cps_df['EMPSTAT'] = cps_df['EMPSTAT'].isin([10, 12]).astype(int)

# Creating treatment indicator that will be used for difference in differences estimation (i.e. 1 for NJ and after 04 1992, 0 otherwise)
cps_df['DID_DUMMY'] = cps_df['POLICY_PERIOD'] * cps_df['NJ_PA']
cps_df

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,FIRMSIZE,INCTOT,INCWAGE,EDATT,EDATTLY,NJ_PA,PERIOD,POLICY_PERIOD,RECESSION_PERIOD,DID_DUMMY
1,1988,23,1,NaN,1.987120e+13,NaN,NaN,1,1537.2900,19871200002201,...,NaN,NaN,NaN,NaN,NaN,1,1988-01,0,1,0
2,1988,92,1,NaN,1.988010e+13,NaN,NaN,1,1266.4100,19880100009201,...,NaN,NaN,NaN,NaN,NaN,1,1988-01,0,1,0
6,1988,103,1,NaN,1.986110e+13,NaN,NaN,1,1184.5300,19861100010201,...,NaN,NaN,NaN,NaN,NaN,1,1988-01,0,1,0
10,1988,138,1,NaN,1.987110e+13,NaN,NaN,1,1254.0600,19871100013701,...,NaN,NaN,NaN,NaN,NaN,1,1988-01,0,1,0
11,1988,138,1,NaN,1.987110e+13,NaN,NaN,2,1340.4600,19871100013703,...,NaN,NaN,NaN,NaN,NaN,1,1988-01,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1519207,1997,48978,12,2051.2907,1.997110e+13,NaN,NaN,2,2051.2907,19971104892702,...,NaN,NaN,NaN,NaN,NaN,0,1997-12,1,0,0
1519209,1997,48978,12,2051.2907,1.997110e+13,NaN,NaN,4,2327.8280,19971104892704,...,NaN,NaN,NaN,NaN,NaN,0,1997-12,1,0,0
1519213,1997,48980,12,2317.6384,1.997110e+13,NaN,NaN,2,2039.9635,19971104892902,...,NaN,NaN,NaN,NaN,NaN,0,1997-12,1,0,0
1519214,1997,48980,12,2317.6384,1.997110e+13,NaN,NaN,3,2142.6876,19971104892903,...,NaN,NaN,NaN,NaN,NaN,0,1997-12,1,0,0


In [31]:
# Creating variable for formatted period that will be used for graph axis
cps_df = cps_df[['PERIOD', 'IND1990', 'EMPSTAT', 'ASECWT', 'WTFINL', 'DID_DUMMY', 'NJ_PA', 'POLICY_PERIOD', 'RECESSION_PERIOD']]
cps_df.to_csv('/Users/chrisgee/UCSD2526/replication_project/data/state_by_month_cps.csv')